# Mục 3e — Gộp năm boolean thành một mã số
**Võ Minh Huy — 2311110117**

Thứ tự cố định: anaemia, diabetes, high_blood_pressure, sex, smoking. Trọng số 1, 2, 4, 8, 16; mã từ 0 đến 31. Mã không phải điểm rủi ro hoặc thang mức độ bệnh. Giữ nhãn DEATH_EVENT riêng.

## 1. Đọc và kiểm tra năm biến, lưu quy ước
Đầu vào là bản đầy đủ 299 hồ sơ sau xử lý nhãn. Nếu một trong năm biến thiếu hoặc không thuộc 0/1, dừng trước mã hóa.

In [5]:
from pathlib import Path
from decimal import Decimal
import pandas as pd

thu_muc = Path.cwd()
if thu_muc.name == "python":
    thu_muc = thu_muc.parent
file_vao = thu_muc / "ket_qua/15_khong_nhat_quan/02_death_event/ban_lam_viec_nhan_python.csv"
ban_day_du = pd.read_csv(file_vao, dtype=str, keep_default_na=False)
cac_cot = ["anaemia", "diabetes", "high_blood_pressure", "sex", "smoking"]
trong_so = [1, 2, 4, 8, 16]
gia_tri = ban_day_du[cac_cot].copy()
for cot in cac_cot:
    gia_tri[cot] = gia_tri[cot].map(lambda x: Decimal(x) if x.strip() else None)
if not gia_tri.isin([0, 1]).all().all():
    raise ValueError("Năm field phải đủ giá trị 0/1 trước mã hóa.")
gia_tri = gia_tri.astype(int)
quy_uoc = pd.DataFrame({"ThuTu":range(1, 6), "ThuocTinh":cac_cot, "TrongSo":trong_so})
quy_uoc

,ThuTu,ThuocTinh,TrongSo
0,1,anaemia,1
1,2,diabetes,2
2,3,high_blood_pressure,4
3,4,sex,8
4,5,smoking,16


## 2. Mã hóa, giải mã và kiểm tra toàn bộ hồ sơ
MaBoolean = anaemia + 2×diabetes + 4×high_blood_pressure + 8×sex + 16×smoking. Giải mã mỗi biến bằng (mã // trọng số) % 2.

In [6]:
ma_boolean = sum(gia_tri[cot] * w for cot, w in zip(cac_cot, trong_so))
# Bảng mã hóa chỉ có khóa và một cột mã thay cho năm biến.
bang_ma = pd.DataFrame({"MaHoSo":ban_day_du["MaHoSo"].astype(int), "MaBoolean":ma_boolean})
giai_ma = pd.DataFrame({cot:(ma_boolean // w) % 2
                       for cot, w in zip(cac_cot, trong_so)})
so_ho_so_sai = int(giai_ma.ne(gia_tri).any(axis=1).sum())
tong_ket = pd.DataFrame({"SoHoSo":[len(bang_ma)], "SoFieldTruoc":[5], "SoFieldSau":[1],
    "MaNhoNhat":[int(ma_boolean.min())], "MaLonNhat":[int(ma_boolean.max())],
    "SoMaXuatHien":[int(ma_boolean.nunique())], "SoHoSoGiaiMaSai":[so_ho_so_sai]})
tong_ket

,SoHoSo,SoFieldTruoc,SoFieldSau,MaNhoNhat,MaLonNhat,SoMaXuatHien,SoHoSoGiaiMaSai
0,299,5,1,0,31,28,0


## 3. Năm ví dụ có mã khác nhau
Chuỗi giải mã theo đúng thứ tự của bảng trọng số. MaHoSo nội bộ có thể khác SQL; đối chiếu tổ hợp và mã, không đối chiếu số thứ tự bệnh nhân.

In [7]:
nhat_ky = pd.concat([bang_ma, gia_tri], axis=1)
nhat_ky["GiaiMaTheoThuTu"] = giai_ma.astype(str).agg("|".join, axis=1)
vi_du = (nhat_ky.sort_values(["MaBoolean", "MaHoSo"])
          .drop_duplicates("MaBoolean").head(5).reset_index(drop=True))
vi_du[["MaHoSo"] + cac_cot + ["MaBoolean", "GiaiMaTheoThuTu"]]

,MaHoSo,anaemia,diabetes,high_blood_pressure,sex,smoking,MaBoolean,GiaiMaTheoThuTu
0,69,0,0,0,0,0,0,0|0|0|0|0
1,50,1,0,0,0,0,1,1|0|0|0|0
2,40,0,1,0,0,0,2,0|1|0|0|0
3,5,1,1,0,0,0,3,1|1|0|0|0
4,62,0,0,1,0,0,4,0|0|1|0|0


**Nhận xét:** 299 hồ sơ giữ nguyên; năm field được biểu diễn bằng một mã, có 28 mã xuất hiện trong miền 0–31. Giải mã đủ 299 hồ sơ khớp năm giá trị đầu vào. Mã lớn hơn không có nghĩa bệnh nặng hơn. Khi tính tương quan, khoảng cách hoặc phân tích các đặc điểm, giải mã về các biến riêng. Dữ liệu đầy đủ được giữ nguyên.

## 4. Lưu quy ước và kết quả riêng
Lưu bảng mã hai cột, nhật ký đối chiếu và tổng kết; không ghi đè bảng đầy đủ.

In [8]:
thu_muc_kq = thu_muc / "ket_qua/17_chuyen_doi/01_gop_5_boolean"
thu_muc_kq.mkdir(parents=True, exist_ok=True)
quy_uoc.to_csv(thu_muc_kq / "quy_uoc_python.csv", index=False, encoding="utf-8-sig")
tong_ket.to_csv(thu_muc_kq / "tong_ket_python.csv", index=False, encoding="utf-8-sig")
bang_ma.to_csv(thu_muc_kq / "bang_ma_python.csv", index=False, encoding="utf-8-sig")
nhat_ky.to_csv(thu_muc_kq / "nhat_ky_python.csv", index=False, encoding="utf-8-sig")
pd.DataFrame({"TrangThai":["Đã lưu quy ước và mã gộp; giữ nguyên dữ liệu đầy đủ."]})

,TrangThai
0,Đã lưu quy ước và mã gộp; giữ nguyên dữ liệu đ...
